In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Definir o nível de otimização do transpilador

<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido com os seguintes requisitos.
Recomendamos o uso dessas versões ou de versões mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Dispositivos quânticos reais estão sujeitos a ruído e erros de porta, portanto otimizar os circuitos para reduzir sua profundidade e contagem de portas pode melhorar significativamente os resultados obtidos ao executar esses circuitos.
A função [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) possui um argumento posicional obrigatório, `optimization_level`, que controla o quanto de esforço o transpilador dedica à otimização dos circuitos. Esse argumento pode ser um inteiro com um dos valores 0, 1, 2 ou 3.
Níveis de otimização mais altos geram circuitos mais otimizados ao custo de tempos de compilação maiores.
A tabela a seguir explica as otimizações realizadas para cada configuração.

<table>
  <thead>
    <tr>
      <th>Nível de Otimização</th>
      <th>Descrição</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Sem otimização: tipicamente usado para caracterização de hardware
        - Tradução básica
        - Layout/Roteamento: `TrivialLayout`, que seleciona os mesmos números de qubits físicos que os virtuais e insere SWAPs para fazer funcionar (usando `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Otimização leve:
        -   Layout/Roteamento: O layout é tentado primeiro com `TrivialLayout`. Se SWAPs adicionais forem necessários, um layout com número mínimo de SWAPs é encontrado usando `SabreSwap`, e então `VF2LayoutPostLayout` é usado para tentar selecionar os melhores qubits no grafo.
        -   `InverseCancellation`
        -   Otimização de portas de 1 qubit
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Otimização média:
          - Layout/Roteamento: Nível de otimização 1 (sem trivial) + heurística otimizada com maior
        profundidade de busca e tentativas da função de otimização. Como `TrivialLayout` não é usado, não há tentativa de usar os mesmos números de qubits físicos e virtuais.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Otimização alta:
        - Nível de otimização 2 + heurística otimizada em layout/roteamento com maior esforço/tentativas
        - Ressíntese de blocos de dois qubits usando a [Decomposição KAK de Cartan](https://arxiv.org/abs/quant-ph/0507171).
        - Passos que quebram a unitariedade:
          * `OptimizeSwapBeforeMeasure`: Move as medições para evitar SWAPs
          * `RemoveDiagonalGatesBeforeMeasure`: Remove portas antes de medições que não afetariam os resultados
      </td>
    </tr>
  </tbody>
</table>

## Nível de otimização em ação
Como portas de dois qubits são tipicamente a fonte mais significativa de erros, podemos quantificar aproximadamente a "eficiência de hardware" da transpilação contando o número de portas de dois qubits no circuito resultante.
Aqui, vamos testar os diferentes níveis de otimização em um circuito de entrada composto por uma unitária aleatória seguida de uma porta SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Usaremos o backend simulado `FakeSherbrooke` nos nossos exemplos. Primeiro, vamos transpilar usando o nível de otimização 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

O circuito transpilado possui seis portas ECR de dois qubits.

Repita para o nível de otimização 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

O circuito transpilado ainda possui seis portas ECR, mas o número de portas de um único qubit foi reduzido.

Repita para o nível de otimização 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Isso produz os mesmos resultados que o nível de otimização 1. Note que aumentar o nível de otimização nem sempre faz diferença.

Repita novamente, com o nível de otimização 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Agora, há apenas três portas ECR. Obtemos esse resultado porque no nível de otimização 3, o Qiskit tenta ressintesizar blocos de dois qubits de portas, e qualquer porta de dois qubits pode ser implementada usando no máximo três portas ECR. Podemos obter ainda menos portas ECR se definirmos `approximation_degree` para um valor menor que 1, permitindo que o transpilador faça aproximações que podem introduzir algum erro na decomposição das portas (veja [Parâmetros comumente usados para transpilação](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Este circuito tem apenas duas portas ECR, mas é um circuito aproximado. Para entender como seu efeito difere do circuito exato, podemos calcular a fidelidade entre o operador unitário que este circuito implementa e o unitário exato. Antes de realizar o cálculo, primeiro reduzimos o circuito transpilado, que contém 127 qubits, para um circuito que contém apenas os qubits ativos, dos quais há dois.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>